# Where do the outcome variables go?

`FoldTransversalHadamard.ipynb` reduces a 19-qubit protocol with 34 symbolic measurement outcomes to a two-spider diagram whose phases mention only 8 of them, and PyZX's own `full_reduce` of the same circuit also shows a picture without most of them. That invites a sharp objection:

> every outcome variable occurs exactly once in the circuit, and no correct rewrite of a single occurrence of a Boolean variable can remove it from the diagram — so either a rewrite is losing variables, or those variables are irrelevant.

This notebook settles it by instrumenting **every rewrite** of both pipelines. `track_variable.py` wraps the rewrite appliers of PyZX (fusion, identity removal, pivot, local complementation, gadget rules, copy, Hopf), the Boolean bialgebra call and the row reduction of `zxqec.gpf`, and `remove_isolated_vertices`; after each single application it records where the chosen variable sits — on which spiders (type, phase, degree) and in which factors of the diagram's scalar `g.scalar`. It patches nothing on disk: the wrappers live inside a `with tracing(name)` block.

**The answer in one line.** No rewrite removes a variable. A ZX-diagram is *graph × scalar*, and a variable can move from the graph into the scalar — an edgeless spider $Z(\pi p)$ *is* the number $1 + e^{i\pi p}$. `zx.draw` shows the graph and not the scalar, which is why variables look lost. `reduce_gpf` reports those factors instead of hiding them in `g.scalar`, and calls them detectors.

The notebook proceeds from a one-gate example (§1) and the general rule (§2) to the rewrite-by-rewrite traces (§3, §4), the fate of all 34 outcomes (§5), and the nine final readouts, which is where the interesting cases are (§6).

In [1]:
import os
import random
import re
import itertools
import numpy as np
import pyzx as zx
from pyzx.circuit import Circuit
from pyzx.circuit.gates import HAD, Reset, Measurement
from pyzx.utils import VertexType
from zxqec.qprogram import QProgram
from zxqec.gpf import reduce_gpf, boolean_variables, evaluate_phase, sample_consistent_assignment
from track_variable import tracing, where          # the tracer; see its docstring

N, BITS = 19, 35
X_ANC, Z_ANC = [10, 11, 12, 13], [14, 15, 16, 17]

def load(name):
    return zx.Circuit.load(os.path.join("circuits", "ft_h", name + ".qasm"))

def with_prefix(circ, gates):
    c = Circuit(N, bit_amount=BITS)
    for gate in gates:
        c.add_gate(gate)
    c.add_circuit(circ)
    return c

p = QProgram(N, bits=BITS, initial_states={**{q: '0' for q in range(1, 10)}, 18: '0', **{a: '+' for a in X_ANC + Z_ANC}})
p.add_circuit(load("ft_h"), name="encode + X checks")
p.add_circuit(load("ft_h_2"), name="transversal H")
p.add_circuit(with_prefix(load("ft_h_3"), [Reset(a) for a in X_ANC]), name="Z checks (code 2)")
p.add_circuit(load("ft_h_3.2"), name="X checks (code 2)")
p.add_permutation([0, 3, 6, 9, 2, 5, 8, 1, 4, 7] + list(range(10, 19)), name="rotate 90°")
p.add_circuit(with_prefix(load("ft_h_5"), [Reset(a) for a in Z_ANC]), name="Z checks", physical=True)
p.add_circuit(with_prefix(load("ft_h_5.2"), [gate for a in X_ANC for gate in (Reset(a), HAD(a))]), name="X checks", physical=True)
p.add_circuit(with_prefix(load("ft_h_6"), [Reset(a) for a in Z_ANC]), name="Z checks + decode", physical=True)

g = p.quantum_graph()                                    # the protocol as one ZX-diagram
names = sorted(boolean_variables(g), key=lambda s: int(s[2:-1]))
measured = {gate.result_symbol: (st.index, gate.target)  # outcome -> (stage, qubit measured)
            for st in p.stages for gate in st.circuit.gates if isinstance(gate, Measurement)}
res = reduce_gpf(g)                                      # the GPF reduction: diagram + detectors
print(f"{len(names)} outcome variables; reduced diagram has {res.graph.num_vertices()} vertices "
      f"and the reduction found {len(res.detectors)} detectors")

34 outcome variables; reduced diagram has 4 vertices and the reduction found 24 detectors


## 1. The question, in a one-gate circuit

Measure one qubit in the Z basis with outcome `c`, once on $|0\rangle$ and once on $|+\rangle$. The two circuits differ by a single Hadamard; the measurement, the variable, and its one occurrence are identical.

In [2]:
def one_measurement(state):
    c = Circuit(1)
    if state == '+':
        c.add_gate(HAD(0))                      # |0> -> |+>
    c.add_gate(Measurement(0, result_symbol='c'))
    c.initialize_qubits([True])                 # start in |0>
    return c.to_graph()

for state in ('0', '+'):
    d = one_measurement(state)
    h = d.copy(); zx.full_reduce(h)
    r = reduce_gpf(d)
    print(f"|{state}> measured in the Z basis, outcome c:")
    print("   PyZX full_reduce -> spiders left:", [str(h.phase(v)) for v in h.vertices() if h.type(v) != VertexType.BOUNDARY],
          "  phase nodes in g.scalar:", [str(pn) for pn in h.scalar.phasenodes])
    print("   reduce_gpf       -> detectors:", [str(x) for x in r.detectors])
    for val in (0, 1):
        amp = abs(complex(d.substitute_variables({'c': val}).to_tensor(preserve_scalar=True)))
        print(f"      amplitude of the branch c={val}: {amp:.3f}")
    print()

|0> measured in the Z basis, outcome c:
   PyZX full_reduce -> spiders left: []   phase nodes in g.scalar: ['c']
   reduce_gpf       -> detectors: ['c']
      amplitude of the branch c=0: 1.414
      amplitude of the branch c=1: 0.000

|+> measured in the Z basis, outcome c:
   PyZX full_reduce -> spiders left: []   phase nodes in g.scalar: []
   reduce_gpf       -> detectors: []
      amplitude of the branch c=0: 1.000
      amplitude of the branch c=1: 1.000



$\langle c|0\rangle = 1 + (-1)^c$: the variable survives as a **scalar factor**, worth $2$ for $c = 0$ and $0$ for $c = 1$ — a detector, since measuring $|0\rangle$ in the Z basis always gives 0.

$\langle c|+\rangle = 1/\sqrt2$ for both values: the variable is **genuinely absent** from the answer. Measuring $|+\rangle$ gives a uniformly random bit that influences nothing.

One gate, one occurrence, and both outcomes of the objection already occur — and PyZX and `reduce_gpf` agree in both cases. So "a variable that occurs once left the picture" is not evidence of a lost rewrite; which of the two happens is decided by the diagram the measurement is attached to, not by how the variable is written.

## 2. The rule

A diagram in parity form is a sum over the solutions of a linear system, weighted by the signs of the outcome phases:

$$[\![ D(\vec c) ]\!] \;\propto\; \sum_{\vec y \in A} (-1)^{\,\vec c \cdot \vec y}, \qquad A = \{\vec y : M\vec y = \vec b\} \subseteq \mathbb{F}_2^{\,n}.$$

Write $A = \vec y_0 + V$, with $V$ the solution space of $M\vec y = 0$. Then

$$\sum_{\vec y \in A} (-1)^{\vec c \cdot \vec y} \;=\; (-1)^{\vec c \cdot \vec y_0} \sum_{\vec v \in V} (-1)^{\vec c \cdot \vec v} \;=\; (-1)^{\vec c \cdot \vec y_0}\,|V|\;\cdot\;[\,\vec c \perp V\,],$$

where $[\,\vec c \perp V\,]$ is $1$ when $\vec c \cdot \vec v = 0$ for every $\vec v \in V$ and $0$ otherwise, because a non-trivial character sums to zero. Three consequences, which are exactly the three fates a variable can have:

1. **the value depends on $\vec c$ only through its class modulo $V^{\perp}$** — outcome vectors differing by an element of $V^\perp$ give the same map, so a reduction keeps one representative per class and expresses the rest through it;
2. **a component of $\vec c$ along $V$ forces a condition** — the sum vanishes unless $\vec c \perp V$, and each generator of $V$ contributes a factor $1 + (-1)^{p}$: a **detector**;
3. **anything in neither is invisible** — a direction that is neither constrained nor part of the surviving representative disappears from the answer: a **gauge** bit.

The fate of a variable is therefore a property of the code, not of the variable: which parities of outcomes the protocol makes deterministic, and which of them couple to the qubits that survive.

## 3. `c[5]` through PyZX's `full_reduce`

`c[5]` is the outcome of the Z check with ancilla `q[11]` in stage 2 (the first round of the dual code, right after the transversal Hadamard); `c[1]` is the X check with the same ancilla in stage 0. In the master graph `c[5]` sits, once, on a degree-one X spider — the outcome leaf of that measurement. Every rewrite that moved it (vertex ids are those of the copy `h`, since `g.copy()` renumbers):

In [3]:
h = g.copy()
with tracing('c[5]') as trace_pyzx:
    zx.full_reduce(h)
print(trace_pyzx)

8 of 546 rewrites touched c[5]:
  1. fuse(82, 100)
      before: spider 100: Z(c[5]), degree 1
      after : spider 82: Z(c[5]), degree 4
  2. pivot(27, 117)
      before: spider 82: Z(c[5]), degree 4
      after : spider 82: Z(c[5]), degree 5
  3. pivot(153, 220)
      before: spider 82: Z(c[5]), degree 5
      after : spider 82: Z(c[5]), degree 6
  4. pivot(38, 124)
      before: spider 82: Z(c[5]), degree 6
      after : spider 82: Z(c[5]), degree 8
  5. pivot(42, 213)
      before: spider 82: Z(c[5]), degree 8
      after : spider 82: Z(c[5]), degree 4
  6. pivot_batch(((256, 28), ([], [0])), ((268, 279), ([], [312])))
      before: spider 82: Z(c[5]), degree 4
      after : spider 82: Z(c[5]), degree 5
  7. pivot_batch(((258, 24), ([], [312])),)
      before: spider 82: Z(c[5]), degree 5
      after : spider 82: Z(c[5]), degree 9
  8. pivot_batch(((25, 256), ([], [0])),)
      before: spider 82: Z(c[5]), degree 9
      after : in g.scalar: 1 + e^(i pi (c[5] + c[1]))
      within i

`to_gh` has turned the X leaf into a Z spider on a Hadamard edge; **fuse** merges it into the measurement's spider; the **pivots** on other pairs only rewire its neighbourhood — the degree changes, the phase does not; the last **pivot** adds the phase `c[1]` of a pivoted spider to it and strips its remaining edges; and **`remove_isolated_vertices`**, called inside that pivot, deletes the edgeless spider and records its phase in `g.scalar`. PyZX prints that scalar with all its factors — one per detector:

In [4]:
print("c[5] at the end:", where(h, 'c[5]'))
print("\nPyZX's scalar of the reduced diagram:\n", h.scalar)

c[5] at the end: {'spiders': {}, 'scalar': ['c[5] + c[1]']}

PyZX's scalar of the reduced diagram:
 sqrt(2)^-141(1+exp((c[9])ipi))(1+exp((c[18])ipi))(1+exp((c[12])ipi))(1+exp((c[20])ipi))(1+exp((c[11])ipi))(1+exp((c[10])ipi))(1+exp((c[17])ipi))(1+exp((c[19])ipi))(1+exp((c[27] + c[28])ipi))(1+exp((c[33] + c[32])ipi))(1+exp((c[7] + c[4])ipi))(1+exp((c[15] + c[4])ipi))(1+exp((c[23] + c[4])ipi))(1+exp((c[5] + c[1])ipi))(1+exp((c[14] + c[1])ipi))(1+exp((c[22] + c[1])ipi))(1+exp((c[34] + c[32] + c[30] + c[31])ipi))(1+exp((c[6] + c[2])ipi))(1+exp((c[16] + c[2])ipi))(1+exp((c[24] + c[2])ipi))(1+exp((c[29] + c[30] + c[26] + c[28])ipi))(1+exp((c[8] + c[3])ipi))(1+exp((c[13] + c[3])ipi))(1+exp((c[21] + c[3])ipi))


The factor $1 + e^{i\pi(c_5 + c_1)}$ is $2$ when $c_5 = c_1$ and $0$ otherwise, so it decides whether the branch exists at all. Checking that on the protocol itself: take an outcome that satisfies every detector, then flip `c[5]` alone, which breaks this one detector and no other.

In [5]:
consistent = sample_consistent_assignment(res.constraints, names, random.Random(0))
assert consistent['c[5]'] == consistent['c[1]']                       # the detector holds
flipped = dict(consistent, **{'c[5]': 1 - consistent['c[5]']})        # breaks only c[5] + c[1] = 0
for label, a in [("c[5] = c[1]  (an outcome that can occur)", consistent),
                 ("c[5] != c[1] (an impossible outcome)   ", flipped)]:
    t = g.substitute_variables(a); zx.full_reduce(t)
    print(f"   {label}: the protocol is the zero map: {np.allclose(t.to_tensor(preserve_scalar=True), 0)}")

   c[5] = c[1]  (an outcome that can occur): the protocol is the zero map: False
   c[5] != c[1] (an impossible outcome)   : the protocol is the zero map: True


**What the factor means physically.** `c[1]` measures the X stabilizer $X_1X_2X_4X_5$ of code 1; the transversal Hadamard maps that operator to $Z_1Z_2Z_4Z_5$, which is exactly what the Z check with the same ancilla measures one stage later as `c[5]`. The two outcomes therefore agree in every run. The rewrites did not delete an occurrence of `c[5]`: they *derived* the identity $c_5 = c_1$, and left it in the only place a diagram can hold such a fact — its scalar.

## 4. The same variable through `reduce_gpf`

In [6]:
with tracing('c[5]') as trace_gpf:
    _ = reduce_gpf(g)
print(trace_gpf)
print("\nconstraints mentioning c[5]:", [str(c) for c in res.constraints if 'c[5]' in str(c)])

5 of 808 rewrites touched c[5]:
  1. fuse(82, 100)
      before: spider 100: X(c[5]), degree 1
      after : spider 82: X(c[5]), degree 4
  2. fuse(82, 318)
      before: spider 82: X(c[5]), degree 4
      after : spider 82: X(c[5] + c[0]), degree 6
  3. fuse(29, 314)
      before: spider 82: X(c[5] + c[0]), degree 6
      after : spider 82: X(c[5] + c[0]), degree 4
  4. fuse(82, 320)
      before: spider 82: X(c[5] + c[0]), degree 4
      after : spider 82: X(c[5] + c[1]), degree 0
  5. remove_isolated_vertices()
      before: spider 82: X(c[5] + c[1]), degree 0
      after : in g.scalar: 1 + e^(i pi (c[5] + c[1]))

constraints mentioning c[5]: ['c[5] + c[1]']


Five events instead of eight, all but the last of them fusions: the outcome's spider is fused with neighbouring spiders whose phases carry `c[0]` and then `c[0] + c[1]`, which leaves it with the phase `c[5] + c[1]` (a third fusion, between two other spiders, only changes its degree). After the last fusion it has no edges, and **`remove_isolated_vertices`** records $1 + e^{i\pi(c_5 + c_1)}$ in the scalar — from where `reduce_gpf` reads it out as the constraint `c[5] + c[1]`. Same endpoint as PyZX, by a shorter route.

## 5. All 34 outcomes

One run of each pipeline, then a look at where every variable ended up. The last column classifies it over $\mathbb{F}_2$ from the detectors alone:

* **frame** — still on a spider of the reduced diagram. These, and only these, change the map.
* **fixed** — the detectors force its value (`c[9] = 0`): a check whose outcome is deterministic.
* **tied** — the detectors equate it to other outcomes (`c[5] = c[1]`). Its own value is still random; what is deterministic is the parity.
* **free** — no detector mentions it.

A variable that is not in the frame does not change the map, whichever of the other three it is.

In [7]:
def gf2_rank(rows):
    rows = [r[:] for r in rows]
    rank = 0
    for col in range(len(names)):
        piv = next((k for k in range(rank, len(rows)) if rows[k][col]), None)
        if piv is None:
            continue
        rows[rank], rows[piv] = rows[piv], rows[rank]
        for k in range(len(rows)):
            if k != rank and rows[k][col]:
                rows[k] = [a ^ b for a, b in zip(rows[k], rows[rank])]
        rank += 1
    return rank

def as_row(vs):
    r = [0] * len(names)
    for v in vs:
        r[names.index(v)] ^= 1
    return r

detector_rows = [as_row(re.findall(r'c\[\d+\]', str(c))) for c in res.detectors]
detector_rank = gf2_rank(detector_rows)
in_span = lambda vs: gf2_rank(detector_rows + [as_row(vs)]) == detector_rank     # implied by the detectors?

h = g.copy(); zx.full_reduce(h)                       # PyZX's reduction, once
frame = {v for u in res.graph.vertices() for v in re.findall(r'c\[\d+\]', str(res.graph.phase(u)))}

def classify(name):
    if name in frame:
        return "frame: part of the Pauli correction, changes the map"
    if in_span([name]):
        return f"fixed: {name} = 0"
    tying = [str(c) for c in res.detectors if name in str(c)]
    if tying:
        return "tied:  " + min(tying, key=len) + " = 0"
    return "free:  no detector mentions it"

print(f"{'outcome':8}{'measured':10}{'full_reduce':14}{'reduce_gpf':13}{'classification'}")
for name in names:
    st, q = measured[name]
    pyzx_where = "on spiders" if where(h, name)['spiders'] else "in scalar"
    ours = "on spiders" if name in frame else ("detector" if any(name in str(c) for c in res.detectors) else "gone")
    print(f"{name:8}{'s%d/q%d' % (st, q):10}{pyzx_where:14}{ours:13}{classify(name)}")

outcome measured  full_reduce   reduce_gpf   classification
c[0]    s0/q0     on spiders    on spiders   frame: part of the Pauli correction, changes the map
c[1]    s0/q11    on spiders    on spiders   frame: part of the Pauli correction, changes the map
c[2]    s0/q10    on spiders    on spiders   frame: part of the Pauli correction, changes the map
c[3]    s0/q13    on spiders    on spiders   frame: part of the Pauli correction, changes the map
c[4]    s0/q12    on spiders    on spiders   frame: part of the Pauli correction, changes the map
c[5]    s2/q11    in scalar     detector     tied:  c[5] + c[1] = 0
c[6]    s2/q10    in scalar     detector     tied:  c[6] + c[2] = 0
c[7]    s2/q12    in scalar     detector     tied:  c[7] + c[4] = 0
c[8]    s2/q13    in scalar     detector     tied:  c[8] + c[3] = 0
c[9]    s3/q14    in scalar     detector     fixed: c[9] = 0
c[10]   s3/q15    in scalar     detector     fixed: c[10] = 0
c[11]   s3/q16    in scalar     detector     fixed: c[1

Every variable that is absent from a picture is present in that diagram's scalar or is gauge; the eight on spiders are the Pauli frame. Note also that the two pipelines may name the same class differently — PyZX's `c[26] + c[28] + c[30]` is our `c[29]` modulo the detectors — which is only a choice of representative and not a disagreement.

Twenty-four detectors of rank 24 leave $34 - 24 = 10$ random bits per run, and the map depends on just two of their parities (the Z frame and the X frame of the reduced diagram).

### Counting, in dimensions rather than names

Naming individual variables is slightly misleading, because which one of a relation is eliminated is a choice of representative. The invariant statement is about the outcome space $\Ftwo^{34}$ and three subspaces of it: the span of the detectors, the span of the frame phases, and everything else.

In [8]:
frame_rows = [as_row(re.findall(r'c\[\d+\]', str(res.graph.phase(v))))
              for v in res.graph.vertices() if res.graph.type(v) != VertexType.BOUNDARY]
forced = gf2_rank(detector_rows)
steering = gf2_rank(detector_rows + frame_rows) - forced

print(f"outcome bits                              : {len(names)}")
print(f"forced by the detectors (rank)            : {forced}")
print(f"free directions, i.e. random bits per run : {len(names) - forced}")
print(f"   of which steer the map (the frame)     : {steering}")
print(f"   gauge: implied by nothing, affect nothing : {len(names) - forced - steering}")

outcome bits                              : 34
forced by the detectors (rank)            : 24
free directions, i.e. random bits per run : 10
   of which steer the map (the frame)     : 2
   gauge: implied by nothing, affect nothing : 8


Read that as a decision rule per direction of the outcome space:

* **does it change the map?** If yes it stays on a spider of the reduced diagram — the Pauli frame, two dimensions here, spanned by $c_0 + c_1 + c_2 + c_3 + c_4$ and $c_{26} + c_{29} + c_{32}$;
* **if not, is it implied?** If yes the corresponding spider loses its last edge and the factor $1 + (-1)^p$ it becomes is reported as a detector — 24 dimensions here. Implied by the *preparation* gives a single-variable detector (*fixed*), implied by an *earlier measurement of the same operator* gives a two-term one (*tied*);
* **otherwise it is gauge** — 8 dimensions here. Genuinely random bits that cancel out of every surviving phase, leaving no detector, because the qubits whose measurement produced them no longer exist.

Nothing is lost in either case: a direction leaves the *graph* either into the diagram's *scalar* (as a detector) or by cancelling entirely (gauge). $24 + 2 + 8 = 34$.

## 6. The nine final readouts

The interesting cases are all in the last stage, which measures the nine data qubits: `c[26]`…`c[34]` include variables of all three kinds.

`c[26]`, `c[29]`, `c[32]` are in the frame and the other six are tied — and none of those six changes the map. Why the nine split this way is the code's doing.

**What that stage is.** It is *not* a syndrome-extraction round: no ancilla is used and the hardware measures no parity. `ft_h_6.qasm` applies `h q[i]; measure q[i]` to each data qubit, i.e. it measures the single-qubit operators $X_1, \dots, X_9$, and the protocol ends there. Nothing computes anything from those nine bits afterwards — not the circuit, and not `zxqec`.

What the reduction does is *discover* that four parities of those nine outcomes are deterministic. Why they are is the following fact about the state, which is an explanation of the relations rather than a step anyone performs: $X_2$ and $X_3$ commute, so measuring them separately also determines their product, and that product is the stabilizer $X_2X_3$, whose value was already fixed — if $X_2$ gives $(-1)^{c_{27}}$ and $X_3$ gives $(-1)^{c_{28}}$ then $X_2X_3$ has the value $(-1)^{c_{27}+c_{28}}$, and the state was in its $+1$ eigenspace. Measuring qubit by qubit is *finer* than measuring the stabilizers: the nine bits carry the stabilizer values, the logical value, and some surplus — and the surplus is exactly what turns out to be gauge. In a real experiment a decoder would use these parities as one more round of syndrome data, which is the usual reason for a final data round; here they simply appear as detectors. This works only for X-type stabilizers; the Z checks do not commute with an X-basis readout and are still measured with ancillas in the same stage (`c[21]`–`c[24]`) before the data is destroyed, which is why no Z check appears among the relations below.

In what follows a name like $X_2X_3$ always denotes an **operator of the code** — the stabilizer that ancilla `q10` measures in the check rounds — never something the final round measures directly. The readouts sit on the grid as

```
 position:   1 2 3        readout:   c26 c27 c28
             4 5 6                   c29 c30 c31
             7 8 9                   c32 c33 c34
```

In [9]:
POS = {f'c[{25 + i}]': i for i in range(1, 10)}                  # readout variable -> grid position
VAR_AT = {i: f'c[{25 + i}]' for i in range(1, 10)}               # and back
X_STABILIZERS = {'X1X2X4X5 (ancilla q11)': {1, 2, 4, 5}, 'X2X3     (ancilla q10)': {2, 3},
                 'X5X6X8X9 (ancilla q12)': {5, 6, 8, 9}, 'X7X8     (ancilla q13)': {7, 8}}
SYNDROME = {'X1X2X4X5 (ancilla q11)': 'c[17]', 'X2X3     (ancilla q10)': 'c[18]',   # the same checks, measured
            'X5X6X8X9 (ancilla q12)': 'c[19]', 'X7X8     (ancilla q13)': 'c[20]'}   # with ancillas one stage earlier
LOGICAL_X = {1, 4, 7}                                            # the left column

def decompose(support):
    # write a set of grid positions as a product of the four check generators, if possible
    for k in range(len(X_STABILIZERS) + 1):
        for combo in itertools.combinations(X_STABILIZERS, k):
            acc = set()
            for label in combo:
                acc ^= X_STABILIZERS[label]
            if acc == set(support):
                return ' * '.join(l.split()[0] for l in combo) if combo else 'the identity'
    return 'not in the group the checks generate'

print("relations the reduction found among the readouts alone:")
for c in res.detectors:
    vs = re.findall(r'c\[\d+\]', str(c))
    if vs and all(v in POS for v in vs):
        supp = sorted(POS[v] for v in vs)
        print(f"   {' + '.join(sorted(vs, key=lambda x: POS[x])):48} positions {str(supp):22} = {decompose(supp)}")
fs = sorted(POS[v] for v in frame if v in POS)
print(f"\nreadouts in the Pauli frame: {' + '.join(sorted((v for v in frame if v in POS), key=lambda x: POS[x]))}"
      f"   positions {str(fs):22} = {decompose(fs)};  the logical X: {set(fs) == LOGICAL_X}")

print("\nand each generator on its own, as a parity of readouts:")
for label, support in X_STABILIZERS.items():
    vs = [VAR_AT[i] for i in sorted(support)]
    print(f"   {label:24} {' + '.join(vs):40} = 0   implied by the detectors: {in_span(vs)}")

print("\nis 'that parity = what the ancilla measured one stage earlier' a consequence of the detectors?")
for label, support in X_STABILIZERS.items():
    vs = [VAR_AT[i] for i in sorted(support)] + [SYNDROME[label]]
    print(f"   {label:24} {' + '.join(vs):46} = 0   implied by the detectors: {in_span(vs)}")
logical = [VAR_AT[i] for i in sorted(LOGICAL_X)]
print(f"   {'logical X readout':24} {' + '.join(logical):46}       implied by the detectors: {in_span(logical)}"
      "   <- free: it is the logical measurement result")

relations the reduction found among the readouts alone:
   c[27] + c[28]                                    positions [2, 3]                 = X2X3
   c[26] + c[27] + c[29] + c[30]                    positions [1, 2, 4, 5]           = X1X2X4X5
   c[32] + c[33]                                    positions [7, 8]                 = X7X8
   c[26] + c[27] + c[29] + c[31] + c[32] + c[34]    positions [1, 2, 4, 6, 7, 9]     = X1X2X4X5 * X5X6X8X9 * X7X8

readouts in the Pauli frame: c[26] + c[29] + c[32]   positions [1, 4, 7]              = not in the group the checks generate;  the logical X: True

and each generator on its own, as a parity of readouts:
   X1X2X4X5 (ancilla q11)   c[26] + c[27] + c[29] + c[30]            = 0   implied by the detectors: True
   X2X3     (ancilla q10)   c[27] + c[28]                            = 0   implied by the detectors: True
   X5X6X8X9 (ancilla q12)   c[30] + c[31] + c[33] + c[34]            = 0   implied by the detectors: True
   X7X8     (ancilla q13)  

The `= X1X2X4X5 * X7X8 * ...` column decomposes each relation into the four check generators: membership is in the
*group* they generate, so a relation may be a product of several of them. The reduction reports a reduced echelon
basis of the relations (each one eliminates a different variable), not the generator basis — which is why the last
relation reads $X_1X_2X_4X_5 \cdot X_5X_6X_8X_9 \cdot X_7X_8$ rather than the plain $X_5X_6X_8X_9$; that generator is
implied all the same, as the last block shows.

So the deterministic parities of the readouts are exactly the X-stabilizer parities, each agreeing with the ancilla measurement of the same check one stage earlier (and those measurements were themselves deterministic, which is why the relations involve the readouts alone); the combination that survives on a spider is exactly the logical $X_L$, which nothing fixes.

Nine bits, four relations, so five independent directions remain — and of those five only one, the logical parity, reaches the output. The other four are **gauge**: real random bits of the run that no surviving system can depend on. (Note that "gauge" describes a *direction*, not a variable: `c[27]` and `c[28]` are tied to each other by `c[27] + c[28] = 0`, each of them is still a random bit, and neither changes the map.)

The rewrites that implement this, for the six readouts that leave the picture:

In [10]:
from collections import Counter
for name in ['c[28]', 'c[30]', 'c[33]', 'c[34]', 'c[27]', 'c[31]']:
    with tracing(name) as t:
        r = reduce_gpf(g)
    rules = Counter(e.rule for e in t.events if e.depth == 0)
    print(f"{name}  {classify(name)}")
    print(f"      {sum(rules.values()):2} rewrites: {dict(rules)}")
    print(f"      left on spiders: {where(r.graph, name)['spiders'] or 'none'};"
          f"   in the scalar's global phase: {name in str(r.graph.scalar.phase)}")

c[28]  tied:  c[28] + c[27] = 0


       8 rewrites: {'fuse': 7, 'remove_isolated_vertices': 1}
      left on spiders: none;   in the scalar's global phase: False


c[30]  tied:  c[30] + c[27] + c[29] + c[26] = 0
       8 rewrites: {'fuse': 7, 'remove_isolated_vertices': 1}
      left on spiders: none;   in the scalar's global phase: False


c[33]  tied:  c[33] + c[32] = 0
       5 rewrites: {'fuse': 4, 'remove_isolated_vertices': 1}
      left on spiders: none;   in the scalar's global phase: False


c[34]  tied:  c[34] + c[31] + c[27] + c[29] + c[26] + c[32] = 0
       7 rewrites: {'fuse': 6, 'remove_isolated_vertices': 1}
      left on spiders: none;   in the scalar's global phase: False


c[27]  tied:  c[28] + c[27] = 0
      19 rewrites: {'fuse': 14, 'boolean_bialgebra': 3, 'remove_isolated_vertices': 2}
      left on spiders: none;   in the scalar's global phase: True


c[31]  tied:  c[34] + c[31] + c[27] + c[29] + c[26] + c[32] = 0
      10 rewrites: {'fuse': 7, 'boolean_bialgebra': 2, 'remove_isolated_vertices': 1}
      left on spiders: none;   in the scalar's global phase: True


Two different mechanisms. `c[28]`, `c[30]`, `c[33]` and `c[34]` are fused with their neighbours until the spider has no edges and becomes the factor $1 + (-1)^p$: each is *eliminated by* its relation. `c[27]` and `c[31]` take more rewrites, including the Boolean bialgebra rule, and their own spiders never become such a factor: they cancel out of every phase that survives, leaving a sign of the Boolean bialgebra rule in the scalar's global phase (`in the scalar's global phase: True` above). They still appear in detectors, through the relations that eliminate `c[28]` and `c[34]`.

That the four gauge directions really do not matter can be checked without any reduction: enumerate **all 32** assignments of the five free readouts, repair the four relations, and compare the resulting maps.

In [11]:
free = ['c[26]', 'c[27]', 'c[29]', 'c[31]', 'c[32]']           # the five free readouts
repair = {'c[28]': {'c[27]'},                                  # each tied readout, from its relation above
          'c[30]': {'c[26]', 'c[27]', 'c[29]'},
          'c[33]': {'c[32]'},
          'c[34]': {'c[26]', 'c[27]', 'c[29]', 'c[31]', 'c[32]'}}

def tensor(a):
    t = g.substitute_variables(a); zx.full_reduce(t)
    return t.to_tensor(preserve_scalar=False)

classes = []                                                   # the distinct maps, with the patterns giving them
for bits in itertools.product([0, 1], repeat=len(free)):
    a = dict(consistent)
    a.update(dict(zip(free, bits)))
    for v, deps in repair.items():
        a[v] = sum(a[d] for d in deps) % 2
    assert all(evaluate_phase(c, a) == 0 for c in res.constraints)        # an outcome that can occur
    t = tensor(a)
    hit = next((cl for cl in classes if zx.compare_tensors(cl['map'], t, preserve_scalar=False)), None)
    if hit is None:
        classes.append({'map': t, 'patterns': [], 'logical': set()})
        hit = classes[-1]
    hit['patterns'].append(bits)
    hit['logical'].add(sum(a[v] for v in ['c[26]', 'c[29]', 'c[32]']) % 2)

print(f"{2 ** len(free)} possible readout patterns give {len(classes)} distinct maps:")
for k, cl in enumerate(classes):
    print(f"   map {k}: {len(cl['patterns']):2} patterns, logical parity c[26]+c[29]+c[32] = {cl['logical']}")
print("\nthe map is a function of the logical parity alone:", all(len(cl['logical']) == 1 for cl in classes))

32 possible readout patterns give 2 distinct maps:
   map 0: 16 patterns, logical parity c[26]+c[29]+c[32] = {0}
   map 1: 16 patterns, logical parity c[26]+c[29]+c[32] = {1}

the map is a function of the logical parity alone: True


32 patterns, 2 maps, split exactly by `c[26] + c[29] + c[32]`. Four of the five free directions change nothing — which is forced, not accidental: after the final round the only quantum system left is `q[18]`, holding the logical qubit, and one qubit can absorb exactly one bit of X-frame information. The other outcomes measure operators that anticommute with Z stabilizers of the code, so their values are uniformly random and correlate with nothing that still exists.

| variable | position | fate | why |
|---|---|---|---|
| `c[26]` | 1 | frame | on the logical column $X_1X_4X_7$ |
| `c[27]` | 2 | tied, no effect | its parity with position 3 is the stabilizer $X_2X_3$, whose value was already 0 |
| `c[28]` | 3 | tied, no effect | the same relation, with `c[27]` |
| `c[29]` | 4 | frame | on the logical column |
| `c[30]` | 5 | tied, no effect | completes the parity that is $X_1X_2X_4X_5$ |
| `c[31]` | 6 | tied, no effect | completes the parity that is $X_5X_6X_8X_9$, modulo the other three |
| `c[32]` | 7 | frame | on the logical column |
| `c[33]` | 8 | tied, no effect | completes the parity that is $X_7X_8$ |
| `c[34]` | 9 | tied, no effect | the same relation as `c[31]` |

Which member of a relation is the one eliminated is the reduction's choice of representative, not a fact about the protocol: PyZX's `full_reduce` keeps `c[28]` and eliminates `c[27]` instead, and both diagrams describe the same map.

## Summary

* A rewrite never removes an occurrence of a Boolean variable. It can leave the graph only by becoming part of the **scalar**, as the factor $1 + (-1)^p$ of an edgeless spider, or by cancelling out of every surviving phase.
* Those factors are the **detectors**: parities of outcomes that are 0 in every run. PyZX keeps them in `g.scalar.phasenodes`; `reduce_gpf` returns them as `res.detectors`, which is the only difference between the two.
* What remains on the spiders is the **Pauli frame**: the outcome dependence a decoder has to act on.
* Everything else is **gauge** — directions in the outcome space that no surviving qubit can depend on, because the measurement that produced them destroyed the system carrying the information.
* To check any of this for a variable of your own: `with tracing('c[13]') as t: reduce_gpf(g)` and print `t`.